# Planners-13 — Mesurer la qualité des plans (benchmark de planificateurs)

**Navigation** : [Index](../../README.md) | [<< HTN](Planners-9-HTN.ipynb) | [LLM Planning >>](../04-NeuroSymbolic/Planners-10-LLM-Planning.ipynb)

***

## Objectifs d'apprentissage

À la fin de ce notebook, vous saurez :

1. **Mesurer** la qualité d'un plan (longueur, coût, nombre d'états explorés, temps)
2. **Comparer** des configurations de Fast Downward (heuristiques admissibles vs non admissibles, A* vs greedy) sur un **domaine non-trivial**
3. **Interpreter** pourquoi le choix de l'heuristique change radicalement l'efficience, à plan optimal égal
4. **Reproduire** une recherche naïve en Python et mesurer l'écart face à un vrai planificateur

### Prérequis

- [Planners-4](../02-Classical/Planners-4-Fast-Downward.ipynb) (Fast Downward) et [Planners-5](../02-Classical/Planners-5-Heuristics.ipynb) (Heuristiques)
- Docker + image `jsboige/coursia-fast-downward`

### Durée estimée : 40 minutes

***

## 1. Pourquoi mesurer la qualité des plans

Dans [Planners-4](../02-Classical/Planners-4-Fast-Downward.ipynb), nous comparions les algorithmes et heuristiques de Fast Downward sur des problèmes **simples** (Blocks World à 3 blocs, Logistics, Ferry). La conclusion honnête de ce carnet était :

> *Pour un problème simple comme Blocks World, les différences sont minimes.*

C'est le piège de l'évaluation sur un cas **dégénéré** : quand le problème est trop facile, tous les planificateurs convergent vers le même résultat et on ne voit **pas** la capacité distinctive de l'outil. Un benchmark n'a de sens que si le problème est **assez riche** pour que le choix de la stratégie change un résultat **mesurable**.

Ce notebook reprend la méthode de Planners-4 (comparer des configurations Fast Downward) et l'applique à un domaine **non-trivial** — un Blocks World à 4 blocs — là où la discrimination devient nette : même plan optimal, mais un **écart net de plusieurs fois** sur le nombre d'états explorés.

## 2. Environnement

On réutilise l'infrastructure de Planners-4 : le conteneur Docker `jsboige/coursia-fast-downward` expose Fast Downward via une API HTTP sur le port 8200. La cellule suivante le vérifie, le démarre, et définit les deux fonctions qui porteront tout le benchmark.

In [1]:
import sys, os, platform, subprocess, time, re, json
import urllib.request, urllib.error
from collections import deque

OS_NAME = platform.system()
IS_WINDOWS = OS_NAME == 'Windows'
print(f"Système: {OS_NAME} | Python: {sys.version_info.major}.{sys.version_info.minor}")

# --- Docker + conteneur Fast Downward ---
def check_docker():
    try:
        r = subprocess.run(['docker', '--version'], capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=10)
        return r.returncode == 0, r.stdout.strip()
    except FileNotFoundError:
        return False, "Docker non installé"
    except Exception as e:
        return False, str(e)

DOCKER_OK, DOCKER_INFO = check_docker()
print(f"Docker: {'OK - ' + DOCKER_INFO if DOCKER_OK else 'NON DISPONIBLE - ' + DOCKER_INFO}")

FAST_DOWNWARD_IMAGE = "jsboige/coursia-fast-downward:latest"
FD_API_PORT = 8200
FD_CONTAINER_NAME = "coursia-fast-downward"

def start_fd_container():
    subprocess.run(['docker', 'rm', '-f', FD_CONTAINER_NAME], capture_output=True, timeout=10)
    r = subprocess.run(['docker', 'run', '-d', '--name', FD_CONTAINER_NAME,
                        '-p', f'{FD_API_PORT}:{FD_API_PORT}', FAST_DOWNWARD_IMAGE],
                       capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=120)
    if r.returncode != 0:
        print("Erreur conteneur:", r.stderr); return False
    for _ in range(20):
        try:
            resp = urllib.request.urlopen(f"http://localhost:{FD_API_PORT}/health", timeout=2)
            if resp.status == 200:
                print(f"Conteneur Fast Downward prêt (port {FD_API_PORT})"); return True
        except (urllib.error.URLError, ConnectionError):
            time.sleep(1)
    print("Timeout: conteneur non réactif"); return False

FD_DOCKER_OK = start_fd_container() if DOCKER_OK else False
if not FD_DOCKER_OK:
    print("Fast Downward Docker non disponible — les mesures FD seront marquées ÉCHEC")

# --- Helper d'exécution Fast Downward via API HTTP ---
def run_fast_downward_docker(domain_pddl, problem_pddl, search_config="astar(lmcut())", timeout=60):
    if not FD_DOCKER_OK:
        return {"success": False, "error": "Fast Downward Docker non disponible", "time": 0}
    payload = json.dumps({"domain": domain_pddl, "problem": problem_pddl, "search": search_config}).encode("utf-8")
    req = urllib.request.Request(f"http://localhost:{FD_API_PORT}/plan", data=payload,
                                 headers={"Content-Type": "application/json"}, method="POST")
    try:
        t0 = time.time()
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            data = json.loads(resp.read().decode("utf-8"))
        elapsed = time.time() - t0
        if data.get("returncode") == 0:
            return {"success": True, "stdout": data.get("stdout", ""), "time": elapsed}
        return {"success": False, "stdout": data.get("stdout", ""), "error": data.get("stderr", ""), "time": elapsed}
    except urllib.error.URLError as e:
        return {"success": False, "error": f"Connexion échouée: {e}", "time": 0}
    except Exception as e:
        return {"success": False, "error": str(e), "time": 0}

# --- Helper d'extraction des mesures ---
def extract_metrics(stdout):
    cost = re.search(r'Plan cost:\s*(\d+)', stdout)
    length = re.search(r'Plan length:\s*(\d+)', stdout)
    evaluated = re.search(r'Evaluated (\d+) state', stdout)
    expanded = re.search(r'Expanded (\d+) state', stdout)
    return {
        "plan_length": int(length.group(1)) if length else None,
        "plan_cost": int(cost.group(1)) if cost else None,
        "evaluated": int(evaluated.group(1)) if evaluated else None,
        "expanded": int(expanded.group(1)) if expanded else None,
    }

print(f"Helper run_fast_downward_docker() prêt | Fast Downward: {'DISPONIBLE' if FD_DOCKER_OK else 'INDISPONIBLE'}")

Système: Windows | Python: 3.13
Docker: OK - Docker version 29.8.2, build 7fc2dff


Conteneur Fast Downward prêt (port 8200)
Helper run_fast_downward_docker() prêt | Fast Downward: DISPONIBLE


### Lecture du résultat

L'audit de l'environnement confirme les outils disponibles (voir le tableau imprimé ci-dessus) :

| Composant | Rôle |
|-----------|------|
| Docker | fait tourner le conteneur Fast Downward |
| Image `coursia-fast-downward` | embarque Fast Downward + API HTTP |
| Port 8200 | endpoint `/plan` (reçoit domaine + problème + config de recherche) |

Le helper `run_fast_downward_docker()` encapsule l'appel HTTP ; `extract_metrics()` en extrait les mesures (longueur, coût, états évalués). Tout le benchmark repose sur ces deux fonctions.

***

## 3. Un domaine non-trivial

Nous reprenons le domaine STRIPS **Blocks World** de Planners-4, mais à une échelle où l'espace d'états devient non-négligeable : une **tour de 4 blocs** à construire. Le domaine est défini une fois pour toutes, puis un petit générateur de problème permet de changer la taille de la tour (ce que les exercices utiliseront).

In [2]:
# Blocks World — domaine STRIPS classique (4 actions)
DOMAIN_BLOCKS = """
(define (domain blocks)
  (:requirements :strips :typing)
  (:types block)
  (:predicates
    (on ?x - block ?y - block)
    (ontable ?x - block)
    (clear ?x - block)
    (handempty)
    (holding ?x - block)
  )
  (:action pick-up
    :parameters (?x - block)
    :precondition (and (clear ?x) (ontable ?x) (handempty))
    :effect (and (not (ontable ?x)) (not (clear ?x)) (not (handempty)) (holding ?x)))
  (:action put-down
    :parameters (?x - block)
    :precondition (holding ?x)
    :effect (and (not (holding ?x)) (clear ?x) (handempty) (ontable ?x)))
  (:action stack
    :parameters (?x - block ?y - block)
    :precondition (and (holding ?x) (clear ?y))
    :effect (and (not (holding ?x)) (not (clear ?y)) (clear ?x) (handempty) (on ?x ?y)))
  (:action unstack
    :parameters (?x - block ?y - block)
    :precondition (and (on ?x ?y) (clear ?x) (handempty))
    :effect (and (not (on ?x ?y)) (not (clear ?x)) (not (handempty)) (holding ?x) (clear ?y)))
)
"""

# Problème NON-TRIVIAL : tour de 4 blocs a/b/c/d (a en haut)
N_BLOCKS = 4
BLOCKS = ['a', 'b', 'c', 'd']

def build_tower_problem(blocks):
    """Problème : construire la tour blocks[0]/blocks[1]/.../blocks[-1] (blocks[0] en haut)."""
    n = len(blocks)
    objs = ' '.join(blocks) + ' - block'
    init = []
    for b in blocks:
        init.append('(ontable ' + b + ')')
        init.append('(clear ' + b + ')')
    init.append('(handempty)')
    init_str = '\n    '.join(init)
    goal_parts = ['(on ' + blocks[i] + ' ' + blocks[i + 1] + ')' for i in range(n - 1)]
    goal = goal_parts[0] if len(goal_parts) == 1 else '(and ' + ' '.join(goal_parts) + ')'
    lines = [
        '(define (problem tower-' + str(n) + ')',
        '  (:domain blocks)',
        '  (:objects ' + objs + ')',
        '  (:init',
        '    ' + init_str,
        '  )',
        '  (:goal ' + goal + ')',
        ')',
    ]
    return '\n'.join(lines) + '\n'

PROBLEM_TOWER4 = build_tower_problem(BLOCKS)
print("Domaine blocks défini (4 actions STRIPS)")
print('Problème: tour de ' + str(N_BLOCKS) + ' blocs — but: ' + ' > '.join(reversed(BLOCKS)))
print('----- PDDL du problème -----')
print(PROBLEM_TOWER4)

Domaine blocks défini (4 actions STRIPS)
Problème: tour de 4 blocs — but: d > c > b > a
----- PDDL du problème -----
(define (problem tower-4)
  (:domain blocks)
  (:objects a b c d - block)
  (:init
    (ontable a)
    (clear a)
    (ontable b)
    (clear b)
    (ontable c)
    (clear c)
    (ontable d)
    (clear d)
    (handempty)
  )
  (:goal (and (on a b) (on b c) (on c d)))
)



### Lecture du résultat

Le problème affiché ci-dessus demande de construire une **tour de 4 blocs** (`a` au sommet, posé sur `b`, sur `c`, sur `d`). C'est le même domaine STRIPS que Planners-4, mais à une **échelle où l'espace d'états devient non-négligeable** : chaque bloc peut être sur la table, sur un autre bloc, ou en main, et l'ordre dans lequel on les empile génère de nombreux états intermédiaires.

C'est précisément cette richesse qui rend le benchmark informatif : sur 3 blocs, tous les planificateurs convergent vite ; sur 4 blocs, la **stratégie de recherche** commence à compter.

***

## 4. Le benchmark — 5 configurations, un même problème

On soumet le **même** problème à cinq configurations de Fast Downward : quatre recherches A* de qualité croissante (`blind`, `hmax`, `ff`, `lmcut`), dont trois heuristiques **admissibles** (`blind`, `hmax`, `lmcut`) qui garantissent chacune un plan optimal — `ff` ne l'est pas en général (plan relaxé : elle peut surestimer le coût restant), mais atteint ici le même optimum, ce qui rend le cas plus riche — et une recherche gloutonne (`lazy_greedy`), qui sacrifie toute garantie d'optimalité au profit de la vitesse. Pour chacune, on mesure la longueur du plan, son coût, le nombre d'états **évalués** et le temps.

In [3]:
# Benchmark : 5 configurations Fast Downward sur le meme probleme non-trivial
SEARCH_CONFIGS = [
    ("A* + blind",    "astar(blind())"),
    ("A* + hmax",     "astar(hmax())"),
    ("A* + lmcut",    "astar(lmcut())"),
    ("A* + ff",       "astar(ff())"),
    ("Greedy + ff",   "lazy_greedy([ff()])"),
]

results = {}
print(f"{'Configuration':<16} {'Succès':<8} {'Longueur':<9} {'Coût':<6} {'Évalués':<9} {'Temps'}")
print("-" * 62)
for name, cfg in SEARCH_CONFIGS:
    r = run_fast_downward_docker(DOMAIN_BLOCKS, PROBLEM_TOWER4, search_config=cfg, timeout=60)
    if r['success']:
        m = extract_metrics(r['stdout'])
        results[name] = m
        print(f"{name:<16} {'OK':<8} {str(m['plan_length']):<9} {str(m['plan_cost']):<6} {str(m['evaluated']):<9} {r['time']:.2f}s")
    else:
        results[name] = None
        print(f"{name:<16} {'ÉCHEC':<8} {r.get('error','')[:32]}")

Configuration    Succès   Longueur  Coût   Évalués   Temps
--------------------------------------------------------------


A* + blind       OK       6         6      121       0.14s


A* + hmax        OK       6         6      56        0.08s
A* + lmcut       OK       6         6      22        0.09s


A* + ff          OK       6         6      26        0.09s


Greedy + ff      OK       10        10     20        0.11s


### Lecture du résultat — là où la discrimination apparaît

Le tableau ci-dessus est le cœur du notebook. Lisez-le en croisant deux colonnes :

- **Longueur / Coût** : les quatre recherches A* (`blind`, `hmax`, `ff`, `lmcut`) trouvent le **même plan optimal** — ces colonnes sont identiques d'une ligne à l'autre. Trois sont admissibles (`blind`, `hmax`, `lmcut`) : l'optimalité en découle. `ff` **n'est pas admissible** en général, mais atteint ici le même optimum — la garantie est une **conjonction** (« A* **et** heuristique admissible »), pas une propriété de la seule recherche.
- **Évalués** : c'est ici que tout se joue. La recherche **aveugle** (`blind`) explore un nombre d'états **très supérieur** à celui de `lmcut`, pour aboutir au **même** plan. L'heuristique ne sert pas à *améliorer* le plan (déjà optimal) ; elle sert à **l'atteindre en explorant moins**.

C'est la leçon que le cas à 3 blocs de Planners-4 masquait : *le choix de l'heuristique est un choix d'EFFICIENCE* (nombre d'états, temps), pas de qualité de plan, dès lors que l'heuristique est admissible. La ligne `Greedy + ff` (`lazy_greedy`) illustre le contrepoint : un plan **non optimal** — 10 actions contre 6 pour l'optimum, soit **4 actions de détour** — pour un nombre d'états évalués comparable à `lmcut` (20 contre 22) : sur cette instance, la vitesse n'est pas acquise, seule la garantie est perdue.

***

## 5. Un vrai planificateur vs une recherche naïve

Pour isoler la valeur de l'heuristique, la cellule suivante reproduit **en Python, sans Fast Downward**, une recherche **aveugle** (BFS) du même problème. C'est le contrepoint le plus pur : même problème, même complétude, mais **aucune heuristique** pour guider l'exploration. On compare alors le nombre d'états que cette recherche naïve explore à celui de `A* + lmcut`.

In [4]:
# --- Représentation d'un état Blocks World ---
# state = (on, holding) où on: dict bloc -> 'table' | 'hand' | bloc ; holding: bloc ou None
def initial_state(blocks):
    on = {b: 'table' for b in blocks}
    return (frozenset(on.items()), None)

def is_clear(on, b):
    return not any(loc == b for loc in on.values())

def successors(state, blocks):
    on_f, holding = state
    on = dict(on_f)
    for x in blocks:
        if holding is None and on[x] == 'table' and is_clear(on, x):   # pick-up
            on2 = dict(on); on2[x] = 'hand'
            yield (frozenset(on2.items()), x)
        if holding == x:                                                # put-down
            on2 = dict(on); on2[x] = 'table'
            yield (frozenset(on2.items()), None)
        if holding == x:                                                # stack
            for y in blocks:
                if y != x and is_clear(on, y):
                    on2 = dict(on); on2[x] = y
                    yield (frozenset(on2.items()), None)
        if holding is None and on[x] not in ('table', 'hand') and is_clear(on, x):  # unstack
            on2 = dict(on); on2[x] = 'hand'
            yield (frozenset(on2.items()), x)

def goal_reached(state, blocks):
    on = dict(state[0])
    for i in range(len(blocks) - 1):
        if on[blocks[i]] != blocks[i + 1]:
            return False
    return True

# --- BFS naïve (sans heuristique) ---
def bfs_solve(blocks):
    start = initial_state(blocks)
    if goal_reached(start, blocks):
        return 0, 0
    queue = deque([start])
    seen = {start}
    expanded = 0
    depth = {start: 0}
    while queue:
        state = queue.popleft()
        expanded += 1
        d = depth[state]
        for nxt in successors(state, blocks):
            if nxt not in seen:
                seen.add(nxt)
                depth[nxt] = d + 1
                if goal_reached(nxt, blocks):
                    return d + 1, expanded
                queue.append(nxt)
    return None, expanded

plan_len_bfs, expanded_bfs = bfs_solve(BLOCKS)
print("BFS naïve (Python, sans heuristique) :")
print('  Longueur du plan trouvé : ' + str(plan_len_bfs))
print('  États explorés          : ' + str(expanded_bfs))
lm = results.get("A* + lmcut")
if lm and lm.get("evaluated"):
    print('\nFast Downward A* + lmcut : longueur ' + str(lm['plan_length']) + ', états évalués ' + str(lm['evaluated']))
    ratio = expanded_bfs / lm['evaluated']
    print('  -> le BFS naïve explore ~' + str(int(round(ratio))) + 'x plus d\'états que A*+lmcut pour le MÊME plan optimal')
else:
    print('\n(Comparaison indisponible : A*+lmcut n\'a pas produit de mesure)')

BFS naïve (Python, sans heuristique) :
  Longueur du plan trouvé : 6
  États explorés          : 95

Fast Downward A* + lmcut : longueur 6, états évalués 22
  -> le BFS naïve explore ~4x plus d'états que A*+lmcut pour le MÊME plan optimal


### Lecture du résultat — un vrai planificateur vs une recherche naïve

La cellule précédente reproduit, en une quarantaine de lignes de Python, une recherche **aveugle** (BFS) du même problème, sans aucune heuristique. Le message est le même que pour Fast Downward : la recherche naïve explore **beaucoup** d'états, là où `A* + lmcut` s'en tient à une fraction.

L'écart affiché (le ratio d'états explorés) est la traduction chiffrée d'une idée simple : **une heuristique admissible est ce qui sépare un planificateur qui « devine bien » d'une recherche qui « essaie tout »**. C'est la valeur distinctive de l'outil SOTA, mesurable — et c'est exactement ce qu'un benchmark sur un problème trop simple ne permet pas de voir.

***

## 6. Exercices

### Exercice 1 — Passer à 5 blocs

Construisez le problème **tower-5** (blocs `a b c d e`) avec `build_tower_problem`, relancez le benchmark (les 5 configurations) et comparez le nombre d'états **évalués** de `A* + blind` à celui de `A* + lmcut`. Le ratio doit **croître** avec la taille de la tour : c'est la signature d'une heuristique qui « travaille ».

In [5]:
# Exercice 1 : passer à 5 blocs
# 1. BLOCKS5 = ['a', 'b', 'c', 'd', 'e']
# 2. PROBLEM_TOWER5 = build_tower_problem(BLOCKS5)
# 3. Relancez la boucle de SEARCH_CONFIGS sur PROBLEM_TOWER5 et mesurez le ratio
#    états(A*+blind) / états(A*+lmcut).
#
# Indice : le but est (and (on a b) (on b c) (on c d) (on d e))
# Indice : réutilisez extract_metrics() sur la sortie de chaque configuration

pass  # TODO étudiant
print("Exercice à compléter")

Exercice à compléter


**Indice :** le but de la tour à 5 blocs est `(and (on a b) (on b c) (on c d) (on d e))`. La longueur optimale est de 8 actions. Ce qui change, c'est le nombre d'états que chaque configuration doit évaluer pour l'atteindre : observez l'écart `blind`/`lmcut`.

### Exercice 2 — Une heuristique admissible en mode glouton

Ajoutez la configuration `greedy + lmcut` (recherche gloutonne avec une heuristique **admissible**) et observez si le plan trouvé reste **optimal en longueur**.

La recherche gloutonne évalue `f(n) = h(n)` **sans** le coût accumulé `g(n)` : l'admissibilité de `h` ne suffit plus à garantir l'optimalité.

In [6]:
# Exercice 2 : greedy + lmcut
# Ajoutez ("Greedy + lmcut", "lazy_greedy([lmcut()])") à SEARCH_CONFIGS,
# relancez le benchmark, et comparez la LONGUEUR du plan à celle de A* + lmcut.
#
# Indice : greedy minimise h(n) seul — un plan plus long peut avoir un h plus faible
# Indice : la question à trancher est « même longueur qu'A*+lmcut ? »

pass  # TODO étudiant
print("Exercice à compléter")

Exercice à compléter


**Indice :** comparez la colonne *Longueur* de `Greedy + lmcut` à celle de `A* + lmcut`. S'ils diffèrent, c'est la preuve que l'admissibilité garantit l'optimalité **pour A***, pas pour le glouton.

### Exercice 3 — Reconstruire la séquence du plan

Modifiez `bfs_solve` pour qu'il renvoie aussi la **séquence d'actions** du plan (et pas seulement sa longueur), puis vérifiez que cette séquence satisfait bien le but.

C'est le passage de la *mesure* (combien d'états) à la *preuve* (quel plan) : un benchmark doit pouvoir produire le plan qu'il compte.

In [7]:
# Exercice 3 : reconstruire la séquence d'actions du BFS
# 1. Dans bfs_solve, conservez le PARENT de chaque état (dict parent[nxt] = state)
# 2. Depuis l'état but, remontez la chaîne parent jusqu'à l'état initial
# 3. Déduisez de chaque transition (parent -> enfant) l'action appliquée
#
# Indice : une transition change un seul bloc (on[x]) et/ou holding — c'est l'action
# Indice : comparez la séquence obtenue au plan que Fast Downward a imprimé

pass  # TODO étudiant
print("Exercice à compléter")

Exercice à compléter


**Indice :** la remontée de `parent` donne la chaîne d'états ; pour en extraire l'action, comparez `on` et `holding` entre un état et son parent (un seul bloc change de support, et `holding` passe de `None` à un bloc ou vice-versa).

***

## 7. Ce qu'il faut retenir

1. **Un benchmark n'a de sens que sur un problème non-trivial** : sur un cas dégénéré, tous les planificateurs convergent et on ne mesure rien.
2. **Admissible = plan optimal garanti** : `blind`, `hmax`, `lmcut` donnent la même longueur de plan ; l'admissibilité protège la qualité.
3. **L'heuristique est un choix d'efficience** : à plan optimal égal, `lmcut` explore nettement moins d'états que `blind`. C'est le rôle de l'heuristique.
4. **Satisficing (greedy) = vitesse sans garantie** : plus rapide à lancer, plan non garanti optimal.
5. **La valeur d'un vrai planificateur se mesure** : face à une recherche naïve (BFS), l'écart d'états explorés est l'indice de sa capacité distinctive.

## Pour aller plus loin

- Pattern databases et heuristiques additives (`merge-and-shrink`) : au-delà de `lmcut`.
- Domaines à coûts explicites (`:metric`) : quand la longueur ne suffit plus.
- [Planners-9-HTN](Planners-9-HTN.ipynb) : la planification hiérarchique, un autre levier d'efficience.

***

**Notebook précédent** : [Planners-9-HTN](Planners-9-HTN.ipynb) · **Suivant** : [Planners-10-LLM-Planning](../04-NeuroSymbolic/Planners-10-LLM-Planning.ipynb)